In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.chdir('/content/drive/MyDrive/Stock')

Mounted at /content/drive


In [2]:
# !wget http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
# !tar -xzvf ta-lib-0.4.0-src.tar.gz
# %cd ta-lib
# !./configure --prefix=/usr
# !make
# !make install
# !pip install Ta-Lib

In [3]:
! pip install TA-Lib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.1 MB/s eta 0:00:00


In [4]:
# import talib

# print(talib.__version__)

In [5]:
# import pandas as pd

# # 使用 FinMind.ipynb 所抓取的基本面資料
# df = pd.read_csv("3231.csv")
# df["date"] = pd.to_datetime(df["date"])
# df = df.sort_values("date").reset_index(drop=True)
# # df = df[df["date"] != "2025-07-30"].reset_index(drop=True)
# df
# # df.info()

# 迴圈

In [6]:
import pandas as pd
import talib

stock_list = ["2308", "2368", "2449", "2454"]

for stock in stock_list:
    try:
        # 動態產生輸入與輸出的檔名
        input_file = f"{stock}.csv"
        output_file = f"{stock}_Cal.csv"

        # 讀取 CSV
        df = pd.read_csv(input_file)

        # 日期處理與排序
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date").reset_index(drop=True)

        # 計算 SMA
        df["SMA5"] = talib.SMA(df["close"], timeperiod=5)
        df["SMA10"] = talib.SMA(df["close"], timeperiod=10)
        df["SMA20"] = talib.SMA(df["close"], timeperiod=20)

        # 計算 KD
        df["K"], df["D"] = talib.STOCH(
            high=df["max"],
            low=df["min"],
            close=df["close"],
            fastk_period=9,
            slowk_period=3,
            slowk_matype=0,   # 0 = SMA
            slowd_period=3,
            slowd_matype=0    # 0 = SMA
        )

        # 計算 RSI
        df["RSI_7"] = talib.RSI(df["close"], timeperiod=7)
        df["RSI_14"] = talib.RSI(df["close"], timeperiod=14)

        # 計算布林通道 (Bollinger Bands)
        df["BB_Upper"], df["BB_Middle"], df["BB_Lower"] = talib.BBANDS(
            df["close"],
            timeperiod=20,
            nbdevup=2,     # 2 倍標準差
            nbdevdn=2,
            matype=0       # 0 = SMA
        )

        # 計算 MACD
        df["MACD"], df["MACD_Signal"], df["MACD_Hist"] = talib.MACD(
            df["close"],
            fastperiod=12,
            slowperiod=26,
            signalperiod=9
        )

        # 儲存檔案
        df.to_csv(output_file, index=False)
        print(f"成功處理並儲存：{output_file}")

    except FileNotFoundError:
        print(f"找不到檔案：{input_file}，已跳過。")
    except Exception as e:
        print(f"處理 {stock} 時發生錯誤：{e}")

成功處理並儲存：2308_Cal.csv
成功處理並儲存：2368_Cal.csv
成功處理並儲存：2449_Cal.csv
成功處理並儲存：2454_Cal.csv


In [7]:
xx

NameError: name 'xx' is not defined

# 移動平均 MA（Moving Average）

常見的移動平均有三種：簡單移動平均（SMA）、加權移動平均（WMA）、指數移動平均（EMA），這邊只示範簡單移動平均（SMA）

In [ ]:
df["SMA5"] = talib.SMA(df["close"], timeperiod=5)
df["SMA10"] = talib.SMA(df["close"], timeperiod=10)
df["SMA20"] = talib.SMA(df["close"], timeperiod=20)
# df["SMA60"] = talib.SMA(df["close"], timeperiod=60)

# df.dropna(inplace=True)
df.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 7))

plt.plot(df["date"], df["close"], label="Close")
plt.plot(df["date"], df["SMA5"], label="SMA5")
plt.plot(df["date"], df["SMA10"], label="SMA10")
plt.plot(df["date"], df["SMA20"], label="SMA20")

plt.title("2330 Moving Average")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
! pip install mplfinance

In [ ]:
import mplfinance as mpf

df["date"] = pd.to_datetime(df["date"])

# mplfinance 需使用OHLCV
plot_df = df.copy()
plot_df = plot_df.rename(columns={
    "open": "Open",
    "max": "High",
    "min": "Low",
    "close": "Close",
    "Trading_Volume": "Volume"})

plot_df = plot_df.set_index("date")

# 畫 K 線 + 均線(使用已經算好的 SMA5、SMA10、SMA20)
apds = [
    mpf.make_addplot(plot_df["SMA5"]),
    mpf.make_addplot(plot_df["SMA10"]),
    mpf.make_addplot(plot_df["SMA20"]),]

mpf.plot(
    plot_df,
    type="candle",
    addplot=apds,
    volume=True,
    style="yahoo",
    figsize=(14, 8),
    title="2330 K-Line with SMA")

# 隨機指標 KD（Stochastic Oscillator）

優點
*   操作簡單，容易上手
*   反應靈敏，擅長震盪盤操作
*   可與多種指標搭配使用

缺點
*   易產生假訊號，特別是在趨勢行情中
*   過於靈敏，可能導致過度交易
*   依賴技術參數設定

可搭配 RSI 指標：強化超買超賣訊號、MACD 指標：濾除雜訊、確認中長期動能、移動平均線（MA）：確認趨勢方向


In [ ]:
df["K"], df["D"] = talib.STOCH(
    high=df["max"],
    low=df["min"],
    close=df["close"],
    fastk_period=9,
    slowk_period=3,
    slowk_matype=0,   # 0 = SMA
    slowd_period=3,
    slowd_matype=0    # 0 = SMA
)

df[["date", "close", "K", "D"]].tail()

# 相對強弱指數 RSI（Relative Strength Index）



可搭配 移動平均線：確認整體方向不變、MACD：增加交易信號、布林通道：把握區間震盪行情中的短線機會

In [ ]:
df["RSI_7"] = talib.RSI(df["close"], timeperiod=7)
df["RSI_14"] = talib.RSI(df["close"], timeperiod=14)

df[["date", "close", "RSI_7", "RSI_14"]].tail()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1,
    figsize=(14, 10),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1, 1]}
)

# 股價
ax1.plot(df["date"], df["close"], label="Close")
ax1.set_title("2330 Price")
ax1.legend()
ax1.grid(True)

# KD
ax2.plot(df["date"], df["K"], label="K")
ax2.plot(df["date"], df["D"], label="D")
ax2.axhline(80, linestyle="--", linewidth=1)
ax2.axhline(20, linestyle="--", linewidth=1)
ax2.set_ylim(0, 100)
ax2.set_ylabel("KD")
ax2.legend()
ax2.grid(True)

# RSI
ax3.plot(df["date"], df["RSI_7"], label="RSI(7)")
ax3.plot(df["date"], df["RSI_14"], label="RSI(14)")
ax3.axhline(70, linestyle="--", linewidth=1)
ax3.axhline(30, linestyle="--", linewidth=1)
ax3.set_ylim(0, 100)
ax3.set_ylabel("RSI")
ax3.set_xlabel("Date")
ax3.legend()
ax3.grid(True)

plt.tight_layout()
plt.show()

# 布林通道 BB（Bollinger Bands）

In [ ]:
df["BB_Upper"], df["BB_Middle"], df["BB_Lower"] = talib.BBANDS(
    df["close"],
    timeperiod=20,
    nbdevup=2,     # 2 倍標準差
    nbdevdn=2,
    matype=0)      # 0 = SMA

df[["date", "close", "BB_Upper", "BB_Middle", "BB_Lower"]].tail()

In [ ]:
# mplfinance 需使用OHLCV
plot_df = df.rename(columns={
    "open": "Open",
    "max": "High",
    "min": "Low",
    "close": "Close",
    "Trading_Volume": "Volume"
}).set_index("date")

# KD
kd80 = mpf.make_addplot([80]*len(plot_df), panel=1, linestyle='--')
kd20 = mpf.make_addplot([20]*len(plot_df), panel=1, linestyle='--')

# RSI
rsi70 = mpf.make_addplot([70]*len(plot_df), panel=2, linestyle='--')
rsi30 = mpf.make_addplot([30]*len(plot_df), panel=2, linestyle='--')

apds = [
    # 布林通道
    mpf.make_addplot(plot_df["BB_Upper"], panel=0),
    mpf.make_addplot(plot_df["BB_Middle"], panel=0),
    mpf.make_addplot(plot_df["BB_Lower"], panel=0),

    # KD
    mpf.make_addplot(plot_df["K"], panel=1, ylabel="KD"),
    mpf.make_addplot(plot_df["D"], panel=1),

    # RSI
    mpf.make_addplot(plot_df["RSI_7"], panel=2, ylabel="RSI"),
    mpf.make_addplot(plot_df["RSI_14"], panel=2, ylabel="RSI"),

    # 超買超賣
    kd80, kd20,
    rsi70, rsi30]

mpf.plot(
    plot_df,
    type="candle",
    addplot=apds,
    volume=False,              # 不顯示成交量
    style="yahoo",
    figsize=(14, 10),
    panel_ratios=(5, 2, 2),
    title="2330 Technical Analysis")

# MACD（Moving Average Convergence / Divergence）

In [ ]:
df["MACD"], df["MACD_Signal"], df["MACD_Hist"] = talib.MACD(
    df["close"],
    fastperiod=12,
    slowperiod=26,
    signalperiod=9)

df[["date", "close", "MACD", "MACD_Signal", "MACD_Hist"]].tail()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1,
    figsize=(14, 10),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1, 1]})

# 股價
ax1.plot(df["date"], df["close"], label="Close")
ax1.set_title("2330 Price")
ax1.legend()
ax1.grid(True)

# MACD
ax2.plot(df["date"], df["MACD"], label="MACD")
ax2.plot(df["date"], df["MACD_Signal"], label="Signal")

colors = ["red" if x >= 0 else "green" for x in df["MACD_Hist"]]
ax2.bar(df["date"], df["MACD_Hist"], color=colors, width=1)

ax2.axhline(0, color="black", linewidth=1)

ax2.set_ylabel("MACD")
ax2.legend()
ax2.grid(True)

# RSI
ax3.plot(df["date"], df["RSI_7"], label="RSI(7)")
ax3.plot(df["date"], df["RSI_14"], label="RSI(14)")
ax3.axhline(70, linestyle="--", linewidth=1)
ax3.axhline(30, linestyle="--", linewidth=1)
ax3.set_ylim(0, 100)

ax3.set_ylabel("RSI")
ax3.set_xlabel("Date")
ax3.legend()
ax3.grid(True)

plt.tight_layout()
plt.show()

# 輸出計算好的指標，接著就要設定交易策略做回測，並計算報酬

In [ ]:
# df.info()
df.to_csv("3231_Cal.csv", index=False)

參考資料

https://www.oanda.com/bvi-ft/lab-education/technical_analysis/use_kd/

https://www.oanda.com/bvi-ft/lab-education/technical_analysis/what_is_rsi/

https://havocfuture.tw/blog/python-indicators-talib

https://zhuanlan.zhihu.com/p/342075180
